# FDS Challenge: My Notebook

## 0. Configuration

In [ ]:
import json
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DEBUG = True


# --- PATHS ---
TRAIN_FILE_PATH = "./dataset/train.jsonl"
TEST_FILE_PATH = "./dataset/test.jsonl"
TYPE_CHART_PATH = "./dataset/type_chart_gen2.csv"
TYPE_MAP_PATH = "type_map.json"
SAMPLE_SUBMISSION_PATH = "./dataset/sample_submission.csv"


# --- FEATURES ---
BAD_STATUSES = {"slp", "frz", "tox", "psn", "par", "brn"}
MULTS_WEIGHTS = {
    0: -3,
    0.25: -2,
    0.5: -1,
    1: 0,
    2: 1,
    4: 2,
}

# --- TRAINING ---
"""
"logistic_regression":
A simple linear model that weights each feature and outputs a win probability.
It is fast and easy to interpret, but it only captures straightforward relationships.
Features are standardized first.

"random_forest":
Trains 150 decision trees on different random subsets of the data and averages their votes.
It handles non-linear patterns well and is robust to noise and overfitting.

"lightgbm":
A gradient boosting model that builds trees one after another, each correcting the errors of the previous ones.
It is usually the most accurate on tabular data.

"ensemble":
A soft-voting combination of the three models above, averaging their probabilities with weights.
It tends to be the most stable, but it is the slowest because it trains three models per fold.
"""
MODEL_TYPE = "ensemble"
RANDOM_STATE = 42
K_FOLDS = 5


# --- BATTLE EXAMPLE ---
def print_battle(data, battle_idx=0):
    """
    Prints the whole battle at index `battle_idx`
    """

    battle = data[battle_idx]
    output_path = f"battle_{battle_idx}.json"
    with open(output_path, "w") as f:
        json.dump(battle, f, indent=4)

    print(f"SUCCESS: battle saved to {output_path}.")

## 1. Loading the Data

In [ ]:
# --- LOAD DATA ---
def load_data(file_path):
    """
    Loads the whole `file_path` and stores it as a list of battles
    """

    print(f"Loading data from '{file_path}'...")
    data = []
    try:
        with open(file_path, "r") as f:
            for line in f:
                data.append(json.loads(line))
        print(f"SUCCESS: loaded {len(data)} battles.")
        if DEBUG:
            print_battle(data)
    except FileNotFoundError:
        print(f"ERROR: Could not find the file at '{file_path}'.")

    return data


# Loading data
train_data = load_data(TRAIN_FILE_PATH)
test_data = load_data(TEST_FILE_PATH)

### Types Charts

In [ ]:
# -- TYPE CHART ---
def load_type_chart(file_path):
    try:
        return pd.read_csv(file_path)
    except FileNotFoundError:
        print(f"ERROR: Could not find the file at '{file_path}'.")


# --- POKEMON TYPE MAP ---
def create_type_map(train_data, test_data, file_path=TYPE_MAP_PATH):
    """
    Build the type map, save it as JSON, and return it
    """

    type_map = {}
    for data in (train_data,test_data):
        for battle in data:
            for pokemon in battle["p1_team_details"]:
                type_map[pokemon["name"]] = pokemon["types"]

            lead = battle["p2_lead_details"]
            type_map[lead["name"]] = lead["types"]

    with open(file_path, "w") as f:
        json.dump(type_map, f, indent=4)
    return type_map


def get_multiplier(move_type: str, pkm_type1: str, pkm_type2: str | None = None) -> float:
    if type_chart is None:
        raise ValueError("Type chart is not loaded.")

    m1 = type_chart.get((move_type, pkm_type1), 1.0) if pkm_type1 else 1.0
    m2 = type_chart.get((move_type, pkm_type2), 1.0) if pkm_type2 else 1.0

    return m1 * m2


def p():
    pass


# Getting type charts
type_chart = load_type_chart(TYPE_CHART_PATH)
df_indexed = type_chart.set_index(["attacking_type", "defending_type"])
type_chart = df_indexed["multiplier"].to_dict()
type_map = create_type_map(train_data,test_data)

## 2. Extract Features (both Train and Test Features)

In [ ]:
# --- BASE STATS ---
def feature_base_stats(battle, features):
    """
    Add base pokemon statistics of:
    - player 1 full team
    - player 2 lead pokemon
    - player 1 mean pokemon stats - player 2 lead pokemon stats
    """

    p1_team = battle["p1_team_details"]
    features["p1_mean_hp"] = np.mean([p["base_hp"] for p in p1_team])
    features["p1_mean_atk"] = np.mean([p["base_atk"] for p in p1_team])
    features["p1_mean_def"] = np.mean([p["base_def"] for p in p1_team])
    features["p1_mean_spa"] = np.mean([p["base_spa"] for p in p1_team])
    features["p1_mean_spd"] = np.mean([p["base_spd"] for p in p1_team])
    features["p1_mean_spe"] = np.mean([p["base_spe"] for p in p1_team])
    features["p1_mean_bst"] = features["p1_mean_hp"] + features["p1_mean_atk"] + features["p1_mean_def"] + features["p1_mean_spa"] + features["p1_mean_spd"] + features["p1_mean_spe"]

    p2_lead = battle["p2_lead_details"]
    features["p2_lead_hp"] = p2_lead["base_hp"]
    features["p2_lead_atk"] = p2_lead["base_atk"]
    features["p2_lead_def"] = p2_lead["base_def"]
    features["p2_lead_spa"] = p2_lead["base_spa"]
    features["p2_lead_spd"] = p2_lead["base_spd"]
    features["p2_lead_spe"] = p2_lead["base_spe"]
    features["p2_lead_bst"] = features["p2_lead_hp"] + features["p2_lead_atk"] + features["p2_lead_def"] + features["p2_lead_spa"] + features["p2_lead_spd"] + features["p2_lead_spe"]

    features["diff_mean_hp"] = features["p1_mean_hp"] - features["p2_lead_hp"]
    features["diff_mean_atk"] = features["p1_mean_atk"] - features["p2_lead_atk"]
    features["diff_mean_def"] = features["p1_mean_def"] - features["p2_lead_def"]
    features["diff_mean_spa"] = features["p1_mean_spa"] - features["p2_lead_spa"]
    features["diff_mean_spd"] = features["p1_mean_spd"] - features["p2_lead_spd"]
    features["diff_mean_spe"] = features["p1_mean_spe"] - features["p2_lead_spe"]
    features["diff_mean_bst"] = features["p1_mean_bst"] - features["p2_lead_bst"]


# --- FAINTED POKEMON ---
def feature_fainted(p1_status_map, p2_status_map, features):
    """
    Add fainted pokemon counts using each pokemon's latest observed status:
    - player 1 fainted pokemon count
    - player 2 fainted pokemon count
    """

    features["p1_fnt"] = sum(1 for status in p1_status_map.values() if status == "fnt")
    features["p2_fnt"] = sum(1 for status in p2_status_map.values() if status == "fnt")


# --- HP LOSS ---
def get_hp_pct_lost(index, turn, player, team):
    """
    Track HP loss from one turn for each observed pokemon:
    - remember its latest HP across switches, starting from full HP
    - accumulate HP loss in its three ten-turn intervals
    Healing reduces HP loss.
    """

    name = turn[f"p{player}_pokemon_state"]["name"]
    hp = turn[f"p{player}_pokemon_state"]["hp_pct"]
    if name not in team:
        team[name] = {"last_hp": 1.0, "losses": [0.0, 0.0, 0.0]}

    pokemon = team[name]
    pokemon["losses"][index // 10] += pokemon["last_hp"] - hp
    pokemon["last_hp"] = hp


def feature_hp_lost(p1_hp_map, p2_hp_map, features):
    """
    Add mean HP loss for both players after all observed turns:
    - sum each pokemon's loss in turns 0-10, 10-20, and 20-30
    - divide each interval total by the final number of observed team pokemon
    Partial intervals (eg. battle ended before 30 turns) are valid too.
    Unobserved intervals remain zero.
    """

    features["p1_hp_pct_lost_0_10"] = sum(p["losses"][0] for p in p1_hp_map.values()) / len(p1_hp_map)
    features["p1_hp_pct_lost_10_20"] = sum(p["losses"][1] for p in p1_hp_map.values()) / len(p1_hp_map)
    features["p1_hp_pct_lost_20_30"] = sum(p["losses"][2] for p in p1_hp_map.values()) / len(p1_hp_map)

    features["p2_hp_pct_lost_0_10"] = sum(p["losses"][0] for p in p2_hp_map.values()) / len(p2_hp_map)
    features["p2_hp_pct_lost_10_20"] = sum(p["losses"][1] for p in p2_hp_map.values()) / len(p2_hp_map)
    features["p2_hp_pct_lost_20_30"] = sum(p["losses"][2] for p in p2_hp_map.values()) / len(p2_hp_map)


# --- BAD STATUSES ---
def get_statuses(turn, p1_status_map, p2_status_map):
    """
    Get:
    - player 1 active pokemon status
    - player 2 active pokemon status
    """

    p1_state = turn["p1_pokemon_state"]
    p1_status_map[p1_state["name"]] = p1_state["status"]

    p2_state = turn["p2_pokemon_state"]
    p2_status_map[p2_state["name"]] = p2_state["status"]


def feature_statuses(p1_status_map, p2_status_map, features):
    """
    Add bad status counts using each pokemon's latest observed status:
    - player 1 bad status count
    - player 2 bad status count
    - player 1 bad status count - player 2 bad status count
    """

    features["p1_bad_status_count"] = sum(1 for status in p1_status_map.values() if status in BAD_STATUSES)
    features["p2_bad_status_count"] = sum(1 for status in p2_status_map.values() if status in BAD_STATUSES)
    features["diff_bad_status"] = features["p1_bad_status_count"] - features["p2_bad_status_count"]


# --- ACTIVE HP ---
def feature_active_hp(turn, features):
    """
    Add active pokemon HP at the end of the observed timeline:
    - player 1 active pokemon HP fraction
    - player 2 active pokemon HP fraction
    """

    features["p1_active_hp"] = turn["p1_pokemon_state"]["hp_pct"]
    features["p2_active_hp"] = turn["p2_pokemon_state"]["hp_pct"]


# --- BOOSTS ---
def feature_boosts(turn, features):
    """
    Add active pokemon stat boosts at the end of the observed timeline:
    - player 1 active pokemon boosts
    - player 2 active pokemon boosts
    """

    features["p1_boost_atk"] = turn["p1_pokemon_state"]["boosts"]["atk"]
    features["p1_boost_def"] = turn["p1_pokemon_state"]["boosts"]["def"]
    features["p1_boost_spa"] = turn["p1_pokemon_state"]["boosts"]["spa"]
    features["p1_boost_spd"] = turn["p1_pokemon_state"]["boosts"]["spd"]
    features["p1_boost_spe"] = turn["p1_pokemon_state"]["boosts"]["spe"]

    features["p2_boost_atk"] = turn["p2_pokemon_state"]["boosts"]["atk"]
    features["p2_boost_def"] = turn["p2_pokemon_state"]["boosts"]["def"]
    features["p2_boost_spa"] = turn["p2_pokemon_state"]["boosts"]["spa"]
    features["p2_boost_spd"] = turn["p2_pokemon_state"]["boosts"]["spd"]
    features["p2_boost_spe"] = turn["p2_pokemon_state"]["boosts"]["spe"]


def get_mults(turn, p1_mults, p2_mults):
    mult1, mult2 = 1, 1

    mv1 = turn["p1_move_details"]
    if mv1 != None:
        mv1_category = mv1["category"]
        if mv1_category != "status":
            mv1_type = mv1["type"]
            pkm2_name = turn["p2_pokemon_state"]["name"]
            pkm2_types = type_map.get(pkm2_name,["notype","notype"])
            mult1 = get_multiplier(mv1_type, pkm2_types[0], pkm2_types[1])
        p1_mults[mult1] += 1

    mv2 = turn["p2_move_details"]
    if mv2 != None:
        mv2_category = mv2["category"]
        if mv2_category != "status":
            mv2_type = mv2["type"]
            pkm1_name = turn["p1_pokemon_state"]["name"]
            # pkm1 types are always known
            pkm1_types = type_map[pkm1_name]
            mult2 = get_multiplier(mv2_type, pkm1_types[0], pkm1_types[1])

    p2_mults[mult2] += 1


def feature_mults(p1_mults, p2_mults, features):
    features["p1_0_mult"] = p1_mults[0]
    features["p1_0.25_mult"] = p1_mults[0.25]
    features["p1_0.5_mult"] = p1_mults[0.5]
    features["p1_1_mult"] = p1_mults[1]
    features["p1_2_mult"] = p1_mults[2]
    features["p1_4_mult"] = p1_mults[4]

    features["p1_mult_wmean"] = sum([p1_mults[x] * MULTS_WEIGHTS[x] for x in [0, 0.25, 0.5, 1, 2, 4]]) / 6

    features["p2_0_mult"] = p2_mults[0]
    features["p2_0.25_mult"] = p2_mults[0.25]
    features["p2_0.5_mult"] = p2_mults[0.5]
    features["p2_1_mult"] = p2_mults[1]
    features["p2_2_mult"] = p2_mults[2]
    features["p2_4_mult"] = p2_mults[4]
    features["p2_mult_wmean"] = sum([p2_mults[x] * MULTS_WEIGHTS[x] for x in [0, 0.25, 0.5, 1, 2, 4]]) / 6



In [ ]:
def extract_features(data):
    """
    Extract the features:
    1. base stats
    2. fainted pokemon counts
    3. HP loss in ten-turn intervals
    4. bad statuses active at the end
    5. active HP at the end
    6. individual stat boosts at the end
    7. Type effectiveness multiplier counts
    """

    print("Extracting features...")
    feature_list = []

    for battle in data:
        features = {}

        # 1. Base stats
        feature_base_stats(battle, features)

        p1_status_map, p2_status_map = {}, {}
        p1_hp_map, p2_hp_map = {}, {}

        p1_mults = {0.0: 0, 0.25: 0, 0.5: 0, 1.0: 0, 2.0: 0, 4.0: 0}
        p2_mults = {0.0: 0, 0.25: 0, 0.5: 0, 1.0: 0, 2.0: 0, 4.0: 0}

        for index, turn in enumerate(battle["battle_timeline"]):
            get_statuses(turn, p1_status_map, p2_status_map)
            get_hp_pct_lost(index, turn, 1, p1_hp_map)
            get_hp_pct_lost(index, turn, 2, p2_hp_map)
            get_mults(turn, p1_mults, p2_mults)

        last_turn = battle["battle_timeline"][-1]

        # 2. Fainted pokemon counts
        feature_fainted(p1_status_map, p2_status_map, features)
        # 3. HP loss in ten-turn intervals
        feature_hp_lost(p1_hp_map, p2_hp_map, features)
        # 4. Bad statuses active at the end
        feature_statuses(p1_status_map, p2_status_map, features)
        # 5. Active HP at the end
        feature_active_hp(last_turn, features)
        # 6. Individual stat boosts at the end
        feature_boosts(last_turn, features)
        # 7. Type effectiveness multiplier counts
        feature_mults(p1_mults, p2_mults, features)

        # Keep the identifier and target for the training and submission cells
        if "battle_id" in battle:
            features["battle_id"] = battle["battle_id"]
        if "player_won" in battle:
            features["player_won"] = int(battle["player_won"])

        feature_list.append(features)

    print(f"SUCCESS: extracted {len(feature_list[0])} features.")
    return pd.DataFrame(feature_list).fillna(0)


# Extracting training and test features
train_df = extract_features(train_data)
print(f"Training features: {train_df.shape}.")
display(train_df.head())

test_df = extract_features(test_data)
print(f"Test features: {test_df.shape}.")
# display(test_df.head())

## 3. Training

### Perfomance Metrics

In [ ]:
def compute_metrics(y_true, y_pred, y_prob):
    return {
        # (TP + TN) / total
        "accuracy": accuracy_score(y_true, y_pred),
        # TP / (TP + FP)
        "precision": precision_score(y_true, y_pred, zero_division=0),
        # TP / (TP + FN)
        "recall": recall_score(y_true, y_pred, zero_division=0),
        # 2 * (precision * recall) / (precision + recall) =
        # 2 * TP / (2 * TP + FP + FN)
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
    }

In [ ]:
# --- BUILD MODEL ---
def build_model(model_type=MODEL_TYPE, random_state=RANDOM_STATE):
    """
    Build an instance of `model_type` with seed `random_state`
    """

    match model_type:
        case "logistic_regression":
            return make_pipeline(
                StandardScaler(),
                LogisticRegression(max_iter=1000, C=1.0, random_state=random_state),
            )

        case "random_forest":
            return RandomForestClassifier(
                n_estimators=150,
                max_depth=8,
                min_samples_split=5,
                random_state=random_state,
                n_jobs=-1,
            )

        case "lightgbm":
            return LGBMClassifier(
                n_estimators=150,
                max_depth=5,
                learning_rate=0.04,
                subsample=0.8,
                colsample_bytree=0.8,
                verbose=-1,
                random_state=random_state,
            )

        case "ensemble":
            clf_lgb = LGBMClassifier(
                n_estimators=150,
                max_depth=5,
                learning_rate=0.04,
                subsample=0.8,
                colsample_bytree=0.8,
                verbose=-1,
                random_state=random_state,
            )
            clf_rf = RandomForestClassifier(
                n_estimators=150,
                max_depth=8,
                random_state=random_state,
                n_jobs=-1,
            )
            clf_lr = make_pipeline(
                StandardScaler(),
                LogisticRegression(max_iter=1000, C=1.0, random_state=random_state),
            )
            return VotingClassifier(
                estimators=[("lgb", clf_lgb), ("rf", clf_rf), ("lr", clf_lr)],
                voting="soft",
                weights=[0.5, 0.3, 0.2],
            )

        case _:
            raise ValueError(f"ERROR: Unavailable model: {model_type}.")


# --- TRAINING ---
def train_and_evaluate_cv(X, y, X_test, model_type=MODEL_TYPE, k=K_FOLDS, random_state=RANDOM_STATE):
    """
    Esegue Stratified K-Fold Cross-Validation:
    1. Calcola e registra le metriche su ciascun fold.
    2. Calcola le predizioni Out-Of-Fold (OOF) per una stima globale non-biased.
    3. Calcola la predizione media sul test set (K-Fold test averaging / ensembling).
    """
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=random_state)

    oof_probs = np.zeros(len(X))
    test_probs = np.zeros(len(X_test))
    fold_metrics_list = []

    print(f"\n{'='*60}")
    print(f"Avvio Stratified {k}-Fold Cross Validation con modello: {model_type.upper()}")
    print(f"{'='*60}")

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = build_model(model_type, random_state=random_state + fold)
        model.fit(X_train, y_train)

        # Predizione sul fold di validazione
        val_prob = model.predict_proba(X_val)[:, 1]
        val_pred = (val_prob >= 0.5).astype(int)
        oof_probs[val_idx] = val_prob

        metrics = compute_metrics(y_val, val_pred, val_prob)
        fold_metrics_list.append(metrics)

        print(f"Fold {fold+1}/{k} -> Accuracy: {metrics['accuracy']:.4f} | " f"F1: {metrics['f1']:.4f} | ROC-AUC: {metrics['roc_auc']:.4f}")

        # Accumula predizione sul test set
        test_probs += model.predict_proba(X_test)[:, 1] / k
    # Statistiche aggregate sui K fold
    mean_acc = np.mean([m["accuracy"] for m in fold_metrics_list])
    std_acc = np.std([m["accuracy"] for m in fold_metrics_list])
    mean_f1 = np.mean([m["f1"] for m in fold_metrics_list])
    mean_auc = np.mean([m["roc_auc"] for m in fold_metrics_list])

    # Performance Out-Of-Fold complessiva
    oof_pred = (oof_probs >= 0.5).astype(int)
    overall_oof_acc = accuracy_score(y, oof_pred)
    overall_oof_auc = roc_auc_score(y, oof_probs)

    print(f"\n{'-'*60}")
    print(f"RISULTATI CV AGGREGATI ({k} FOLDS):")
    print(f"Accuracy Media : {mean_acc:.4f} (+/- {std_acc:.4f})")
    print(f"F1 Media       : {mean_f1:.4f}")
    print(f"ROC-AUC Media  : {mean_auc:.4f}")
    print(f"OOF Accuracy   : {overall_oof_acc:.4f}")
    print(f"OOF ROC-AUC    : {overall_oof_auc:.4f}")
    print(f"{'-'*60}\n")

    test_predictions = (test_probs >= 0.5).astype(int)
    return test_predictions, test_probs


# Selezione delle feature escludendo ID e target
feature_cols = [c for c in train_df.columns if c not in ["battle_id", "player_won"]]
print(f"\nNumero di feature utilizzate: {len(feature_cols)}")
print(f"Elenco feature: {feature_cols}")

X_train = train_df[feature_cols]
y_train = train_df["player_won"]
X_test = test_df[feature_cols]

# 3. Addestramento, Validazione K-Fold e Predizione Ensembled
test_predictions, test_probabilities = train_and_evaluate_cv(
    X=X_train,
    y=y_train,
    X_test=X_test,
    model_type=MODEL_TYPE,
    k=K_FOLDS,
    random_state=RANDOM_STATE,
)

## 4. Submission File

In [ ]:
def predict_and_save(predictions, test_battle_ids, output_path="submission.csv"):
    """
    Formatta e salva il file di sottomissione secondo il template richiesto.
    """
    print(f"Salvataggio delle predizioni in '{output_path}'...")
    sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
    expected_columns = ["battle_id", "player_won"]
    if list(sample_submission.columns) != expected_columns:
        raise ValueError(f"Expected submission columns {expected_columns}, got {list(sample_submission.columns)}")

    predictions_by_id = pd.Series(predictions, index=test_battle_ids)
    submission_df = sample_submission.copy()
    submission_df["player_won"] = submission_df["battle_id"].map(predictions_by_id)

    if submission_df["player_won"].isna().any():
        raise ValueError("Alcuni battle_id nella sample submission non hanno una predizione corrispondente.")

    submission_df["player_won"] = submission_df["player_won"].astype(int)
    submission_df.to_csv(output_path, index=False)

    print(f"File '{output_path}' creato con successo!")
    print(submission_df.head(10))
    print(f"Distribuzione predizioni (0 vs 1): {dict(submission_df['player_won'].value_counts())}")


# 4. Creazione del file di sottomissione
predict_and_save(test_predictions, test_battle_ids=test_df["battle_id"], output_path="submission.csv")